# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZohaibArshadNoor/Flyrank-Internship-ML-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook establishes and empirically verifies the Data Contract for Lane 2 (Content Refresh / Opportunity Scoring), defining the unit of analysis, time window, field classification, verification queries, 5-feature frame, and the deliberate target leakage trap.

## 1. Unit of analysis + time window

### Plain-Words Contract (5 Core Answers)
1. **Unit of Analysis (What one row means)**: One row = one unique pseudonymized content item (page grain identified by `content_id`).
2. **Table(s) Used**: `data/raw/content_refresh_anonymized.csv` (starter dataset table representing a mid-panel performance snapshot across 32 clients).
3. **Time Window**: Trailing 90-day observation window (`impressions_90d`, `sessions_90d`) for content items with `content_age_days >= 90` up to 564 days.
4. **What I Predict / Rank (Label or Proxy)**: `is_declining_label` (`trend_direction == 'down'`), predicting traffic decline trajectory to rank a prioritized content refresh review queue.
5. **Deliberately Excluded Field & Why**: Excluded `trend_pct` and `trend_direction` from the feature set because `is_declining_label` is derived directly from `trend_pct`. Including them causes catastrophic 100% target leakage.

In [1]:
import duckdb
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
con = duckdb.connect()

print(f"Data Contract Loaded: {len(df):,} rows x {df.shape[1]} columns")
print("Unit of Analysis: 1 row = 1 content item (content_id)")
print("Time Window: Trailing 90-day snapshot (content_age_days >= 90)")


Data Contract Loaded: 30,000 rows x 44 columns
Unit of Analysis: 1 row = 1 content item (content_id)
Time Window: Trailing 90-day snapshot (content_age_days >= 90)


## 2. Fields: feature / label / context / excluded

### Field Classification Buckets

* **Features (Safe Pre-Decision Signals)**: 
  * `days_since_last_update` (content staleness in days)
  * `impressions_90d` (trailing 90-day search impression volume)
  * `avg_position` (average Google Search Console rank position)
  * `ctr` (click-through rate percentage $\text{clicks}/\text{impressions} \times 100$)
  * `engagement_rate` (GA4 user session engagement percentage)
  * `word_count` (page word length)
  * `content_age_days` (total days since content creation)

* **Label / Proxy**: 
  * `is_declining_label` (1 = declining, 0 = non-declining, computed via `trend_direction == 'down'`)

* **Context (Grouping & Joins - Never Model Inputs)**: 
  * `content_id` (unit join key)
  * `client_id` (client holdout validation grouping key)

* **Excluded (Target Leakage & Privacy)**: 
  * `trend_pct` & `trend_direction`: **Excluded because the target label is derived from them.**
  * Raw queries/URLs/titles: **Excluded for privacy & pseudonymization compliance.**

In [2]:
# Create label and partition feature sets
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

feature_cols = ["days_since_last_update", "impressions_90d", "avg_position", "ctr", "engagement_rate", "word_count", "content_age_days"]
context_cols = ["content_id", "client_id"]
excluded_cols = ["trend_direction", "trend_pct"]

print("=== FIELD CLASSIFICATION VERIFIED ===")
print(f"Safe Features ({len(feature_cols)}): {feature_cols}")
print(f"Context Columns ({len(context_cols)}): {context_cols}")
print(f"Excluded Leakage Columns ({len(excluded_cols)}): {excluded_cols}")


=== FIELD CLASSIFICATION VERIFIED ===
Safe Features (7): ['days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'engagement_rate', 'word_count', 'content_age_days']
Context Columns (2): ['content_id', 'client_id']
Excluded Leakage Columns (2): ['trend_direction', 'trend_pct']


## 3. Verify it with queries (grain, counts, missing values, windows)

### Verification Queries (3 SQL Proofs in DuckDB)

1. **Query 1 — The Grain Proof**: Prove one row really is one unique content item (`content_id`).
2. **Query 2 — Row Count & Date/Age Span**: Verify total slice row count and content age min/max dates.
3. **Query 3 — Availability Check with `IS TRUE`**: Filter with `IS TRUE` to measure how many rows survive minimum visibility criteria (`impressions_90d >= 100`).

### 5-Feature Frame ("Knowable at Decision Moment")
* `days_since_last_update`: **Knowable at decision moment because** content publish/update timestamps are recorded in the CMS prior to review.
* `impressions_90d`: **Knowable at decision moment because** historical search impressions are logged in GSC over the preceding 90-day observation window.
* `avg_position`: **Knowable at decision moment because** average search rank is reported by GSC over the prior observation window.
* `ctr`: **Knowable at decision moment because** click-through rate is calculated from GSC performance data preceding the decision point.
* `engagement_rate`: **Knowable at decision moment because** user session engagement is recorded by GA4 over the prior 90-day window.

### The Leakage Trap Experiment
We deliberately add `trend_pct` to the 5-feature set to demonstrate target leakage, watch Precision@50 jump to **1.000**, and then remove it to retain our honest **0.660** score.

In [3]:
from sklearn.tree import DecisionTreeClassifier

# --- QUERY 1: Grain Check ---
print("--- QUERY 1: Grain Verification (Duplicate content_id Check) ---")
q1 = con.execute("SELECT content_id, COUNT(*) as c FROM df GROUP BY content_id HAVING c > 1 LIMIT 5").fetchall()
print(f"Duplicates found (should be empty []): {q1}")
assert len(q1) == 0, "Grain check failed!"
print("✓ Grain Verified: 1 row = 1 unique content item.\n")

# --- QUERY 2: Row Count & Age Span ---
print("--- QUERY 2: Row Count & Content Age Span ---")
q2 = con.execute("SELECT COUNT(*) as total_rows, MIN(content_age_days) as min_age_days, MAX(content_age_days) as max_age_days FROM df").fetchall()
print(f"Row Count & Age Span: {q2}")
print(f"✓ Verified: {q2[0][0]:,} rows, content age spanning from {q2[0][1]} to {q2[0][2]} days.\n")

# --- QUERY 3: Availability Check with IS TRUE ---
print("--- QUERY 3: Availability Check with IS TRUE ---")
q3 = con.execute("SELECT COUNT(*) as total_rows, COUNT(CASE WHEN (impressions_90d >= 100) IS TRUE THEN 1 END) as visible_rows, AVG(CASE WHEN (impressions_90d >= 100) IS TRUE THEN 1.0 ELSE 0.0 END) as availability_rate FROM df").fetchall()
print(f"Availability Filter Output: Total={q3[0][0]:,}, Surviving Visible Rows={q3[0][1]:,}, Availability Rate={q3[0][2]:.4f}")
print(f"✓ Availability Verified: {q3[0][1]:,} rows ({q3[0][2]*100:.1f}%) survive the (impressions_90d >= 100) IS TRUE filter.\n")

# --- 5-FEATURE FRAME & LEAKAGE TRAP EXPERIMENT ---
print("--- 5-FEATURE FRAME & LEAKAGE TRAP EXPERIMENT ---")
features_5 = ["days_since_last_update", "impressions_90d", "avg_position", "ctr", "engagement_rate"]
X_honest = df[features_5].fillna(0)
y = df["is_declining_label"].values

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# 1. Honest 5-Feature Model
dt_honest = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_honest, y)
score_honest = dt_honest.predict_proba(X_honest)[:, 1]
p50_honest = precision_at_k(score_honest, y, 50)

# 2. Leaky Model (Adding trend_pct on purpose)
X_leaky = df[features_5 + ["trend_pct"]].fillna(0)
dt_leaky = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_leaky, y)
score_leaky = dt_leaky.predict_proba(X_leaky)[:, 1]
p50_leaky = precision_at_k(score_leaky, y, 50)

print(f"Honest 5-Feature Model Precision@50: {p50_honest:.3f}")
print(f"Leaky Model (+trend_pct) Precision@50: {p50_leaky:.3f}  <- TRAP SPRUNG!")
print("\n✓ Leakage Lesson: Adding trend_pct artificially inflates Precision@50 to 1.000 because the label is derived from trend_pct. Removing trend_pct retains our honest 0.660 metric.")


--- QUERY 1: Grain Verification (Duplicate content_id Check) ---
Duplicates found (should be empty []): []
✓ Grain Verified: 1 row = 1 unique content item.

--- QUERY 2: Row Count & Content Age Span ---
Row Count & Age Span: [(30000, 90, 564)]
✓ Verified: 30,000 rows, content age spanning from 90 to 564 days.

--- QUERY 3: Availability Check with IS TRUE ---
Availability Filter Output: Total=30,000, Surviving Visible Rows=22,006, Availability Rate=0.7335
✓ Availability Verified: 22,006 rows (73.4%) survive the (impressions_90d >= 100) IS TRUE filter.

--- 5-FEATURE FRAME & LEAKAGE TRAP EXPERIMENT ---
Honest 5-Feature Model Precision@50: 0.660
Leaky Model (+trend_pct) Precision@50: 1.000  <- TRAP SPRUNG!

✓ Leakage Lesson: Adding trend_pct artificially inflates Precision@50 to 1.000 because the label is derived from trend_pct. Removing trend_pct retains our honest 0.660 metric.


## 4. Data limits

### Named Limitation of this Data Slice
**Trailing Aggregate Snapshot Limitation**: 
The starter dataset slice is a 90-day aggregate snapshot rather than a daily time-series panel. Because feature windows (trailing 90-day impressions/clicks) and label windows (`trend_direction`) are measured over the exact same trailing 90-day observation period, `is_declining_label` functions as a trailing trajectory proxy label rather than a forward-looking predictive outcome label.

To model true forward prediction (features from prior 90 days $\rightarrow$ outcome over next 30 days), work must transition to the Hugging Face full warehouse daily facts (`fact_content_daily_performance`), where feature and target time windows can be strictly separated.

In [4]:
# Data Limitation Summary Verification
print("=== DATA LIMITATION SUMMARY ===")
print("Limitation: Trailing 90-day snapshot overlap (proxy label vs forward-window label).")
print("Mitigation for Capstone: Use Hugging Face daily facts (fact_content_daily_performance) for strict past->future window separation.")


=== DATA LIMITATION SUMMARY ===
Limitation: Trailing 90-day snapshot overlap (proxy label vs forward-window label).
Mitigation for Capstone: Use Hugging Face daily facts (fact_content_daily_performance) for strict past->future window separation.


## Self-check

Before submitting, confirmed each line:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/w03_data_contract.ipynb`.